In [15]:
# Pustaka tambahan (pandas, requests, bs4, lxml sudah tersedia di Colab)
%pip -q install python-dotenv

In [16]:
# ===== 0.1 IMPORT DAN IDENTITAS =====
import os, re, io, csv, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from urllib import robotparser
from dotenv import load_dotenv

# --- WAJIB DIISI ---
NAMA = "Angelina Florensiana Jelita"
NIM = "2501010371"
EMAIL_KAMPUS = "mahasiswa@instiki.ac.id"
TANGGAL_UNDUH_BPS = "2026-11-17"
URL_DOMESTIK = "https://bali.bps.go.id/id/statistics-table/1/MjkjMQ%3D%3D/banyaknya-wisatawan-domestik-bulanan-ke-bali-2004-2025.html"
URL_MANCANEGARA = "https://bali.bps.go.id/id/statistics-table/1/MjEjMQ%3D%3D/banyaknya-wisatawan-mancanegara-bulanan-ke-bali--1982-2008.html"

# Email kampus wajib ada karena dipakai pada User-Agent
if "ISI_" in EMAIL_KAMPUS or "@" not in EMAIL_KAMPUS:
    raise ValueError("Isi EMAIL_KAMPUS dengan email kampus Anda terlebih dahulu.")
if not URL_MANCANEGARA.startswith("http"):
    print("PERINGATAN: URL_MANCANEGARA belum diisi. Provenance akan tidak lengkap.")

# Aturan etika untuk semua request web: timeout, User-Agent berisi email kampus, dan jeda
USER_AGENT = f"DAM-UTS-INSTIKI/1.0 (tugas kuliah; kontak: {EMAIL_KAMPUS})"
HEADERS = {"User-Agent": USER_AGENT}
TIMEOUT = 30          # detik
JEDA = 1.5            # detik antar-request
WITA = dt.timezone(dt.timedelta(hours=8))

_ = load_dotenv()     # membaca .env bila ada; key TIDAK ditulis di notebook
print("Konfigurasi OK:", NAMA, "-", NIM)

Konfigurasi OK: Angelina Florensiana Jelita - 2501010371


In [3]:

from pathlib import Path

# ===== 0.2 STRUKTUR FOLDER =====

RAW = Path("data/raw")
RAW_FILE = RAW / "file_bps"
RAW_API = RAW / "api"
RAW_SCRAPE = RAW / "scraping"

PROC = Path("data/processed")
META = Path("data/metadata")

# Membuat folder
folders = [
    RAW_FILE,
    RAW_API,
    RAW_SCRAPE,
    PROC,
    META
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# Membuat .gitignore
Path(".gitignore").write_text(
    ".env\n*.key\n__pycache__/\n"
)

print("Struktur folder berhasil dibuat!")

for folder in folders:
    print("OK:", folder)


Struktur folder berhasil dibuat!
OK: data/raw/file_bps
OK: data/raw/api
OK: data/raw/scraping
OK: data/processed
OK: data/metadata


In [13]:
# ===== A. DEFINISI NAMA BERKAS =====

NAMA_DOM = "Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025.csv"

NAMA_MAN = "Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008.csv"

FILE_CHECKLIST = "data_ethics_checklist_draf.csv"

print("NAMA_DOM:", NAMA_DOM)
print("NAMA_MAN:", NAMA_MAN)

NAMA_DOM: Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025.csv
NAMA_MAN: Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008.csv


In [17]:

# ============================================================
# 0.3 FUNGSI PROVENANCE DAN PENCATATAN DATASET BPS
# ============================================================

import hashlib
import datetime as dt
import pandas as pd
from pathlib import Path

# ===== 1. PERSIAPAN FOLDER =====

RAW_FILE = Path("data/raw/file_bps")
RAW_FILE.mkdir(parents=True, exist_ok=True)

PROVENANCE = []


# ===== 2. FUNGSI MD5 =====

def md5_berkas(path, ukuran_chunk=1 << 20):
    """Hitung hash MD5 berkas secara bertahap."""
    h = hashlib.md5()

    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(ukuran_chunk), b""):
            h.update(blok)

    return h.hexdigest()


# ===== 3. FUNGSI WAKTU =====

def sekarang():
    try:
        return dt.datetime.now(WITA).isoformat(timespec="seconds")
    except NameError:
        return dt.datetime.now().isoformat(timespec="seconds")


# ===== 4. FUNGSI PENCATATAN PROVENANCE =====

def catat_provenance(
    id_sumber, metode, judul, url, path,
    lisensi, atribusi, tanggal_akses, catatan=""
):
    """Mencatat provenance setiap berkas mentah."""

    p = Path(path)

    # Toleransi perbedaan nama file
    if not p.exists():
        nama_alternatif = p.name.replace("__", ", ").replace("_", " ")
        p_alt = p.parent / nama_alternatif

        if p_alt.exists():
            p = p_alt

    if not p.exists():
        raise FileNotFoundError(
            f"Berkas tidak ditemukan: {p}"
        )

    # Hindari pencatatan ganda
    PROVENANCE[:] = [
        r for r in PROVENANCE
        if r["path"] != str(p)
    ]

    PROVENANCE.append({
        "id_sumber": id_sumber,
        "metode": metode,
        "judul": judul,
        "url": url,
        "tanggal_akses": tanggal_akses,
        "lisensi": lisensi,
        "atribusi": atribusi,
        "nama_berkas": p.name,
        "path": str(p),
        "ukuran_byte": p.stat().st_size,
        "md5": md5_berkas(p),
        "catatan": catatan
    })


# ===== 5. KETERANGAN LISENSI =====

LISENSI_BPS = (
    "Ketentuan penggunaan BPS: perlu diverifikasi berdasarkan "
    "ketentuan portal dan dataset sumber. Cantumkan kredit BPS "
    "sesuai ketentuan yang berlaku. Bukan asumsi lisensi "
    "Creative Commons."
)


# ===== 6. MENCARI FILE CSV BPS SECARA OTOMATIS =====

def cari_file_bps(kata_kunci):

    daftar_file = list(RAW_FILE.glob("*.csv"))

    cocok = [
        f for f in daftar_file
        if all(k.lower() in f.name.lower() for k in kata_kunci)
    ]

    if len(cocok) == 0:
        raise FileNotFoundError(
            f"File dengan kata kunci {kata_kunci} "
            f"tidak ditemukan di folder {RAW_FILE}. "
            "Silakan periksa dan unggah file CSV BPS."
        )

    if len(cocok) > 1:
        raise ValueError(
            f"Ditemukan lebih dari satu file: {cocok}. "
            "Periksa nama file terlebih dahulu."
        )

    return cocok[0]



# ===== 7. IDENTIFIKASI KEDUA DATASET =====

FILE_DOM = RAW_FILE / NAMA_DOM
FILE_MAN = RAW_FILE / NAMA_MAN

print("File domestik    :", FILE_DOM)
print("File mancanegara :", FILE_MAN)

assert FILE_DOM.is_file(), "File domestik tidak ditemukan!"
assert FILE_MAN.is_file(), "File mancanegara tidak ditemukan!"

print("Kedua file CSV berhasil diverifikasi.")



# ===== 8. VERIFIKASI VARIABEL SUMBER =====

variabel_wajib = [
    "URL_DOMESTIK",
    "URL_MANCANEGARA"
]

for nama in variabel_wajib:
    if nama not in globals():
        raise NameError(
            f"Variabel {nama} belum didefinisikan. "
            "Periksa cell konfigurasi sumber BPS sebelumnya."
        )

# Menggunakan tanggal akses yang sudah ditentukan
if "TANGGAL_UNDUH_BPS" in globals():
    TANGGAL_AKSES = TANGGAL_UNDUH_BPS
else:
    TANGGAL_AKSES = dt.datetime.now().strftime("%Y-%m-%d")


# ===== 9. MENCATAT DATASET WISATAWAN DOMESTIK =====

catat_provenance(
    id_sumber="S1-domestik",
    metode="unduh berkas",
    judul="Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025",
    url=URL_DOMESTIK,
    path=FILE_DOM,
    lisensi=LISENSI_BPS,
    atribusi=(
        f'BPS Provinsi Bali. '
        f'"Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025". '
        f'{URL_DOMESTIK}. Diakses {TANGGAL_AKSES}.'
    ),
    tanggal_akses=TANGGAL_AKSES,
    catatan=(
        "Sumber data pada berkas: Hasil Survei "
        "Dinas Pariwisata Provinsi Bali."
    )
)


# ===== 10. MENCATAT DATASET WISATAWAN MANCANEGARA =====

catat_provenance(
    id_sumber="S1-mancanegara",
    metode="unduh berkas",
    judul="Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008",
    url=URL_MANCANEGARA,
    path=FILE_MAN,
    lisensi=LISENSI_BPS,
    atribusi=(
        f'BPS Provinsi Bali. '
        f'"Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008". '
        f'{URL_MANCANEGARA}. Diakses {TANGGAL_AKSES}.'
    ),
    tanggal_akses=TANGGAL_AKSES,
    catatan="Sumber data pada berkas: BPS Provinsi Bali."
)


# ===== 11. MENAMPILKAN HASIL PROVENANCE =====

df_provenance = pd.DataFrame(PROVENANCE)

print("\n========================================")
print("PROVENANCE DATASET BPS BERHASIL DICATAT")
print("========================================")

print("Jumlah dataset:", len(df_provenance))

display(
    df_provenance[
        [
            "id_sumber",
            "nama_berkas",
            "ukuran_byte",
            "md5"
        ]
    ]
)

print("\nPemeriksaan selesai.")


File domestik    : data/raw/file_bps/Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025.csv
File mancanegara : data/raw/file_bps/Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008.csv
Kedua file CSV berhasil diverifikasi.

PROVENANCE DATASET BPS BERHASIL DICATAT
Jumlah dataset: 2


,id_sumber,nama_berkas,ukuran_byte,md5
0,S1-domestik,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ...",3569,c0be2f368c6339e80ee7128a9ea6f55f
1,S1-mancanegara,Banyaknya Wisatawan Mancanegara Bulanan ke Bal...,3887,190204ab068c979ae27ac17d96097e70



Pemeriksaan selesai.


In [18]:

from pathlib import Path

print("=== PEMERIKSAAN FOLDER ===")
print("Lokasi kerja:", Path.cwd())

folder = Path("data/raw/file_bps")

print("Folder tersedia:", folder.exists())

if folder.exists():
    print("\nDaftar file CSV:")
    for file in folder.glob("*.csv"):
        print("-", file.name)
else:
    print("Folder belum ditemukan.")

print("\n=== PEMERIKSAAN VARIABEL ===")
print("URL_DOMESTIK tersedia:", "URL_DOMESTIK" in globals())
print("URL_MANCANEGARA tersedia:", "URL_MANCANEGARA" in globals())
print("WITA tersedia:", "WITA" in globals())
print("TANGGAL_UNDUH_BPS tersedia:", "TANGGAL_UNDUH_BPS" in globals())


=== PEMERIKSAAN FOLDER ===
Lokasi kerja: /content
Folder tersedia: True

Daftar file CSV:
- Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025.csv
- Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008.csv

=== PEMERIKSAAN VARIABEL ===
URL_DOMESTIK tersedia: True
URL_MANCANEGARA tersedia: True
WITA tersedia: True
TANGGAL_UNDUH_BPS tersedia: True


In [19]:
# ===== 1.1 PEMBACAAN DENGAN PARAMETER YANG TEPAT (blok pertama, sebagai demonstrasi) =====
# Menentukan path file asli
path_dom = RAW_FILE / NAMA_DOM

# Toleransi jika nama file asli yang diunggah menggunakan spasi alih-alih underscore
if not path_dom.exists():
    nama_alternatif = path_dom.name.replace("__", ", ").replace("_", " ")
    path_alt = path_dom.parent / nama_alternatif
    if path_alt.exists():
        path_dom = path_alt

blok1 = pd.read_csv(
    path_dom,
    sep=",",                 # pemisah kolom koma
    encoding="utf-8-sig",    # berkas diawali BOM UTF-8
    skiprows=3,              # lewati 2 baris judul + 1 baris kosong
    nrows=14,                # 12 bulan + baris total (BALI) + baris pertumbuhan
    dtype=str,               # tahan sebagai teks karena angka berspasi
    na_values=["-", ""],     # tanda "-" (pertumbuhan 2004) dianggap kosong
)
print(blok1.shape)
blok1.head(14)

(14, 10)


,Bulan,2004,2005,2006,2007,2008,2009,2010,2011,2012
0,Januari,167 106,174 515,202 857,181 266,225 955,264 915,349 575,280 588,333 199
1,Pebruari,133 660,161 808,161 413,144 425,190 792,204 419,238 789,340 508,305 934
2,Maret,118 369,194 411,171 795,161 009,221 181,255 203,202 995,358 313,307 616
3,April,129 730,174 033,192 182,165 509,206 631,247 100,396 898,385 228,331 378
4,Mei,142 186,190 855,188 152,183 736,226 339,289 635,421 369,463 452,525 076
5,Juni,167 718,201 990,204 284,214 957,256 448,304 213,455 456,568 264,569 635
6,Juli,212 463,254 264,245 909,244 032,329 362,340 610,489 307,573 103,524 334
7,Agustus,171 034,217 782,218 117,217 822,259 511,280 972,377 570,440 751,661 334
8,September,168 420,218 382,196 167,181 846,205 304,352 257,594 662,609 633,572 359
9,Oktober,150 827,168 684,238 721,299 724,306 112,330 337,391 722,526 302,667 703


In [20]:
# ===== 1.2 PARSER LENGKAP: SEMUA BLOK TAHUN -> FORMAT LONG =====
BULAN_NO = {"januari": 1, "pebruari": 2, "februari": 2, "maret": 3, "april": 4, "mei": 5, "juni": 6,
            "juli": 7, "agustus": 8, "september": 9, "oktober": 10, "nopember": 11, "november": 11, "desember": 12}
NAMA_BULAN = {1: "Januari", 2: "Februari", 3: "Maret", 4: "April", 5: "Mei", 6: "Juni", 7: "Juli",
              8: "Agustus", 9: "September", 10: "Oktober", 11: "November", 12: "Desember"}

def bersihkan_angka(x):
    """Mengubah angka berformat ribuan menjadi numerik."""
    s = re.sub(r"\s", "", str(x))
    return np.nan if s in ("", "-", "nan") else float(s)

def baca_bps_lebar(path):
    """Kembalikan (data_bulanan_long, kontrol) dari berkas BPS bergaya laporan."""
    mentah = pd.read_csv(path, header=None, names=range(20), dtype=str, encoding="utf-8-sig",
                         sep=",", skip_blank_lines=False, keep_default_na=False)
    kolom0 = mentah[0].astype(str)
    # setiap blok tahun diawali baris yang kolom pertamanya "Bulan ..."
    awal = [i for i, v in kolom0.items() if v.strip().lower().startswith("bulan")]
    data, kontrol = [], []
    for k, h in enumerate(awal):
        akhir = awal[k + 1] if k + 1 < len(awal) else len(mentah)
        blok = mentah.iloc[h:akhir]
        tahun_kolom = [(j, int(v.strip())) for j, v in blok.iloc[0, 1:].items() if str(v).strip().isdigit()]
        for _, baris in blok.iloc[1:].iterrows():
            # "M a r e t   /   M a r c h" -> "maret" (buang spasi, ambil bagian bahasa Indonesia)
            label = re.sub(r"\s", "", str(baris[0]).split("/")[0]).lower()
            if label in BULAN_NO:
                for j, th in tahun_kolom:
                    v = bersihkan_angka(baris[j])
                    if not np.isnan(v):                      # kolom tahun kosong (mis. 2026-2030) diabaikan
                        data.append((th, BULAN_NO[label], v))
            elif label.startswith(("bali", "jumlah")):
                kontrol += [(th, "total_resmi", bersihkan_angka(baris[j])) for j, th in tahun_kolom]
            elif label.startswith("pertumbuhan"):
                kontrol += [(th, "pertumbuhan_resmi", bersihkan_angka(baris[j])) for j, th in tahun_kolom]
    df = pd.DataFrame(data, columns=["tahun", "bulan", "jumlah_wisatawan"])
    df["nama_bulan"] = df["bulan"].map(NAMA_BULAN)
    kt = pd.DataFrame(kontrol, columns=["tahun", "jenis", "nilai"]).pivot(index="tahun", columns="jenis", values="nilai")
    return df, kt

# Menyelesaikan masalah jalur file yang menggunakan spasi alih-alih underscore
path_dom = RAW_FILE / NAMA_DOM
if not path_dom.exists():
    nama_alt_dom = path_dom.name.replace("__", ", ").replace("_", " ")
    if (path_dom.parent / nama_alt_dom).exists():
        path_dom = path_dom.parent / nama_alt_dom

path_man = RAW_FILE / NAMA_MAN
if not path_man.exists():
    nama_alt_man = path_man.name.replace("__", ", ").replace("_", " ")
    if (path_man.parent / nama_alt_man).exists():
        path_man = path_man.parent / nama_alt_man

dom, kontrol_dom = baca_bps_lebar(path_dom)
man, kontrol_man = baca_bps_lebar(path_man)

def lengkapi(df, jenis, sumber):
    df = df.copy()
    df["jenis_wisatawan"] = jenis
    df["sumber_dataset"] = sumber
    return df[["tahun", "bulan", "nama_bulan", "jenis_wisatawan", "jumlah_wisatawan", "sumber_dataset"]]

# Transformasi 1: wide -> long. Transformasi 2: concat dua jenis wisatawan.
gabungan = pd.concat([lengkapi(dom, "domestik", NAMA_DOM), lengkapi(man, "mancanegara", NAMA_MAN)], ignore_index=True)
gabungan["tahun"] = gabungan["tahun"].astype("int64")
gabungan["bulan"] = gabungan["bulan"].astype("int64")
gabungan["jumlah_wisatawan"] = gabungan["jumlah_wisatawan"].astype("int64")
print("domestik:", dom.shape, "| mancanegara:", man.shape, "| gabungan:", gabungan.shape)
gabungan.head()

domestik: (264, 4) | mancanegara: (324, 4) | gabungan: (588, 6)


,tahun,bulan,nama_bulan,jenis_wisatawan,jumlah_wisatawan,sumber_dataset
0,2004,1,Januari,domestik,167106,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ..."
1,2005,1,Januari,domestik,174515,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ..."
2,2006,1,Januari,domestik,202857,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ..."
3,2007,1,Januari,domestik,181266,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ..."
4,2008,1,Januari,domestik,225955,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ..."


In [21]:
# ===== 1.3 IDENTIFIKASI DATASET =====
def dimensi_fisik(path):
    """Jumlah baris fisik dan jumlah kolom maksimum pada berkas mentah."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        baris = list(csv.reader(f))
    return len(baris), max(len(b) for b in baris)

ident = []
for nama, df, jenis in [(NAMA_DOM, dom, "domestik"), (NAMA_MAN, man, "mancanegara")]:
    p = RAW_FILE / nama
    # Gunakan toleransi nama file dengan spasi jika file dengan underscore tidak ditemukan
    if not p.exists():
        nama_alt = p.name.replace("__", ", ").replace("_", " ")
        if (p.parent / nama_alt).exists():
            p = p.parent / nama_alt

    n_baris_fisik, n_kolom_fisik = dimensi_fisik(p)
    ident.append({
        "dataset": jenis,
        "jenis_data": "Terstruktur (tabular CSV, skema tetap: bulan x tahun)",
        "baris_x_kolom_berkas_mentah": f"{n_baris_fisik} x {n_kolom_fisik}",
        "baris_x_kolom_setelah_long": f"{df.shape[0]} x {df.shape[1]}",
        "satuan": "orang (kunjungan wisatawan per bulan); pertumbuhan dalam persen",
        "cakupan_wilayah": "Provinsi Bali (agregat provinsi)",
        "cakupan_waktu": f"{df['tahun'].min()}-{df['tahun'].max()} (bulanan)",
        "ukuran_berkas_byte": p.stat().st_size,
    })
ident = pd.DataFrame(ident).set_index("dataset").T
display(ident)

print("\nTipe data tabel gabungan:")
print(gabungan.dtypes)
print("\nCek jumlah baris >= 100 :", len(gabungan) >= 100, f"({len(gabungan)} observasi bulanan)")

dataset,domestik,mancanegara
jenis_data,"Terstruktur (tabular CSV, skema tetap: bulan x...","Terstruktur (tabular CSV, skema tetap: bulan x..."
baris_x_kolom_berkas_mentah,53 x 10,38 x 15
baris_x_kolom_setelah_long,264 x 4,324 x 4
satuan,orang (kunjungan wisatawan per bulan); pertumb...,orang (kunjungan wisatawan per bulan); pertumb...
cakupan_wilayah,Provinsi Bali (agregat provinsi),Provinsi Bali (agregat provinsi)
cakupan_waktu,2004-2025 (bulanan),1982-2008 (bulanan)
ukuran_berkas_byte,3569,3887



Tipe data tabel gabungan:
tahun                int64
bulan                int64
nama_bulan          object
jenis_wisatawan     object
jumlah_wisatawan     int64
sumber_dataset      object
dtype: object

Cek jumlah baris >= 100 : True (588 observasi bulanan)


In [22]:
# ===== 1.4 ESTIMASI STORAGE 1 TAHUN =====
# Rumus soal: ukuran berkas x 12 bulan, ditambah cadangan 30%.
FAKTOR_CADANGAN = 0.30

paths_terpakai = []
for n in [NAMA_DOM, NAMA_MAN]:
    p = RAW_FILE / n
    if not p.exists():
        nama_alt = p.name.replace("__", ", ").replace("_", " ")
        if (p.parent / nama_alt).exists():
            p = p.parent / nama_alt
    paths_terpakai.append(p)

ukuran_data = sum(p.stat().st_size for p in paths_terpakai)
tahunan = ukuran_data * 12
dengan_cadangan = tahunan * (1 + FAKTOR_CADANGAN)
print(f"Ukuran 2 berkas dataset : {ukuran_data:,} byte ({ukuran_data/1024:.2f} KB)")
print(f"x 12 bulan              : {tahunan:,} byte ({tahunan/1024:.2f} KB)")
print(f"+ cadangan 30%          : {dengan_cadangan:,.0f} byte ({dengan_cadangan/1024:.2f} KB = {dengan_cadangan/1024**2:.4f} MB)")

Ukuran 2 berkas dataset : 7,456 byte (7.28 KB)
x 12 bulan              : 89,472 byte (87.38 KB)
+ cadangan 30%          : 116,314 byte (113.59 KB = 0.1109 MB)


In [38]:
# ===== 1.4b ASUMSI ESTIMASI STORAGE =====
ASUMSI = [
    "Ukuran dasar = jumlah ukuran 2 berkas CSV BPS (domestik + mancanegara).",
    "Rumus soal: ukuran berkas x 12 bulan (diasumsikan 1 berkas baru per bulan dengan ukuran sama).",
    "Cadangan 30% untuk versi data, hasil olahan, dan log (faktor 1,3).",
    "Pertumbuhan ukuran berkas per bulan dianggap konstan; tidak memperhitungkan kompresi.",
]
for i, a in enumerate(ASUMSI, 1):
    print(f"Asumsi {i}: {a}")
print(f"\nHasil: {ukuran_data:,} B x 12 x 1,3 = {dengan_cadangan:,.0f} B = {dengan_cadangan/1024:.2f} KB")

Asumsi 1: Ukuran dasar = jumlah ukuran 2 berkas CSV BPS (domestik + mancanegara).
Asumsi 2: Rumus soal: ukuran berkas x 12 bulan (diasumsikan 1 berkas baru per bulan dengan ukuran sama).
Asumsi 3: Cadangan 30% untuk versi data, hasil olahan, dan log (faktor 1,3).
Asumsi 4: Pertumbuhan ukuran berkas per bulan dianggap konstan; tidak memperhitungkan kompresi.

Hasil: 7,456 B x 12 x 1,3 = 116,314 B = 113.59 KB


In [23]:
# ===== 2.1 PEMERIKSAAN PII DAN QUASI-IDENTIFIER =====
POLA_KOLOM_PII = re.compile(r"nama_orang|nik|ktp|paspor|passport|email|telp|phone|hp|alamat|address|lat|lon|ip_", re.I)
POLA_NILAI = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "telepon": re.compile(r"(?<!\d)(\+62|0)8\d{8,11}(?!\d)"),
    "nik_16_digit": re.compile(r"(?<!\d)\d{16}(?!\d)"),
}

# (a) pola pada nama kolom tabel olahan
kolom_pii = [c for c in gabungan.columns if POLA_KOLOM_PII.search(c)]
# (b) pola pada isi berkas mentah
temuan_nilai = {}
for nama in [NAMA_DOM, NAMA_MAN]:
    p = RAW_FILE / nama
    if not p.exists():
        nama_alt = p.name.replace("__", ", ").replace("_", " ")
        if (p.parent / nama_alt).exists():
            p = p.parent / nama_alt
    teks = p.read_text(encoding="utf-8-sig")
    temuan_nilai[nama] = {k: len(p_re.findall(teks)) for k, p_re in POLA_NILAI.items()}

hasil_pii = {"kolom_terindikasi_pii": kolom_pii, "temuan_pola_pada_isi_berkas": temuan_nilai}
print(json.dumps(hasil_pii, indent=2, ensure_ascii=False))
print("Kolom tabel gabungan:", list(gabungan.columns))

# (c) quasi-identifier: atribut konteks yang jika dikombinasikan dapat mengidentifikasi individu
min_sel = int(gabungan["jumlah_wisatawan"].min())
qi = pd.DataFrame([
    ["tahun", "quasi-identifier (konteks waktu)", "Rendah", "Agregat tahunan"],
    ["bulan / nama_bulan", "quasi-identifier (konteks waktu)", "Rendah", "Agregat bulanan"],
    ["jenis_wisatawan", "quasi-identifier (kategori)", "Rendah", "Hanya 2 kategori besar (domestik/mancanegara)"],
    ["jumlah_wisatawan", "nilai agregat", "Rendah", f"Nilai terkecil {min_sel:,} orang, jadi tidak ada sel kecil yang berisiko re-identifikasi"],
    ["sumber_dataset", "metadata", "Tidak ada", "Nama berkas"],
], columns=["kolom", "kategori", "risiko", "alasan"])
qi

{
  "kolom_terindikasi_pii": [],
  "temuan_pola_pada_isi_berkas": {
    "Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025.csv": {
      "email": 0,
      "telepon": 0,
      "nik_16_digit": 0
    },
    "Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008.csv": {
      "email": 0,
      "telepon": 0,
      "nik_16_digit": 0
    }
  }
}
Kolom tabel gabungan: ['tahun', 'bulan', 'nama_bulan', 'jenis_wisatawan', 'jumlah_wisatawan', 'sumber_dataset']


,kolom,kategori,risiko,alasan
0,tahun,quasi-identifier (konteks waktu),Rendah,Agregat tahunan
1,bulan / nama_bulan,quasi-identifier (konteks waktu),Rendah,Agregat bulanan
2,jenis_wisatawan,quasi-identifier (kategori),Rendah,Hanya 2 kategori besar (domestik/mancanegara)
3,jumlah_wisatawan,nilai agregat,Rendah,"Nilai terkecil 10,141 orang, jadi tidak ada se..."
4,sumber_dataset,metadata,Tidak ada,Nama berkas


In [24]:
# ===== 2.2 DRAF DATA ETHICS CHECKLIST 9 BUTIR =====
path_cek = META / FILE_CHECKLIST
if path_cek.exists():
    cek = pd.read_csv(path_cek, encoding="utf-8-sig")     # pakai berkas Anda bila sudah diunggah
else:
    butir = ["Sumber dan tujuan pengumpulan jelas", "Lisensi dan izin penggunaan diperiksa",
             "Legalitas dan kepatuhan akses", "Identifikasi PII (data pribadi langsung)",
             "Penilaian quasi-identifier", "Minimisasi data",
             "Keamanan penyimpanan dan kredensial", "Akurasi, kualitas, dan keterbatasan data",
             "Transparansi dan provenance"]
    cek = pd.DataFrame({"No": range(1, 10), "Butir pemeriksaan": butir, "Status": "Draf (diisi di 7.3)"})
    cek.to_csv(path_cek, index=False, encoding="utf-8-sig")
print(f"{len(cek)} butir dimuat.")
cek[["No", "Butir pemeriksaan", "Status"]]

9 butir dimuat.


,No,Butir pemeriksaan,Status
0,1,Sumber dan tujuan pengumpulan jelas,Draf (diisi di 7.3)
1,2,Lisensi dan izin penggunaan diperiksa,Draf (diisi di 7.3)
2,3,Legalitas dan kepatuhan akses,Draf (diisi di 7.3)
3,4,Identifikasi PII (data pribadi langsung),Draf (diisi di 7.3)
4,5,Penilaian quasi-identifier,Draf (diisi di 7.3)
5,6,Minimisasi data,Draf (diisi di 7.3)
6,7,Keamanan penyimpanan dan kredensial,Draf (diisi di 7.3)
7,8,"Akurasi, kualitas, dan keterbatasan data",Draf (diisi di 7.3)
8,9,Transparansi dan provenance,Draf (diisi di 7.3)


In [27]:
# ===== 3.1 FUNGSI REQUEST DENGAN TIMEOUT, STATUS CODE, DAN RETRY =====
LOG_REQUEST = []   # bukti penanganan status code

def get_dengan_retry(url, params, label, maks_percobaan=4):
    """GET dengan timeout, User-Agent, backoff eksponensial untuk 429/5xx, dan pencatatan status."""
    for percobaan in range(1, maks_percobaan + 1):
        t0 = time.time()
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
        except (requests.Timeout, requests.ConnectionError) as e:
            LOG_REQUEST.append({"label": label, "percobaan": percobaan, "status": type(e).__name__, "detik": round(time.time() - t0, 2)})
            time.sleep(2 ** percobaan)
            continue
        LOG_REQUEST.append({"label": label, "percobaan": percobaan, "status": r.status_code, "detik": round(time.time() - t0, 2)})
        if r.status_code == 200:
            return r
        if r.status_code in (429, 500, 502, 503, 504):      # sementara: tunggu lalu ulangi
            try:
                tunggu = int(r.headers.get("Retry-After", 2 ** percobaan))
            except ValueError:
                tunggu = 2 ** percobaan
            time.sleep(tunggu)
            continue
        r.raise_for_status()                                  # 4xx lain (mis. 400): jangan diulang
    raise RuntimeError(f"Gagal setelah {maks_percobaan} percobaan: {label}")

In [28]:
# ===== 3.2 AMBIL DATA OPEN-METEO PER BLOK 4 TAHUN =====
URL_OPENMETEO = "https://archive-api.open-meteo.com/v1/archive"
LOKASI = {"nama": "Denpasar, Bali", "lat": -8.65, "lon": 115.2167}
TAHUN_AWAL, TAHUN_AKHIR = 1982, 2025

for th in range(TAHUN_AWAL, TAHUN_AKHIR + 1, 4):
    mulai, selesai = f"{th}-01-01", f"{min(th + 3, TAHUN_AKHIR)}-12-31"
    berkas = RAW_API / f"openmeteo_denpasar_{mulai[:4]}_{selesai[:4]}.json"
    if not berkas.exists():                                   # gunakan cache bila sudah diunduh
        params = {"latitude": LOKASI["lat"], "longitude": LOKASI["lon"], "start_date": mulai, "end_date": selesai,
                  "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Asia/Makassar"}
        resp = get_dengan_retry(URL_OPENMETEO, params, f"openmeteo {mulai[:4]}-{selesai[:4]}")
        berkas.write_bytes(resp.content)                      # simpan respons mentah tanpa diubah
        akses = sekarang()
        time.sleep(JEDA)                                      # jeda antar-request
    else:
        akses = dt.datetime.fromtimestamp(berkas.stat().st_mtime, WITA).isoformat(timespec="seconds")
    catat_provenance("S2-openmeteo", "API", "Open-Meteo Historical Weather API (ERA5) - Denpasar",
                     URL_OPENMETEO, berkas, "CC BY 4.0 (data); API gratis untuk non-komersial",
                     "Weather data by Open-Meteo.com (https://open-meteo.com/), lisensi CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Data diubah: harian diagregasi menjadi bulanan.", akses,
                     f"lat={LOKASI['lat']}, lon={LOKASI['lon']}, {mulai} s.d. {selesai}")
    print("OK", berkas.name)

log = pd.DataFrame(LOG_REQUEST)
print("\nRingkasan status request (kosong bila semua dari cache):")
display(log.groupby("status").size() if len(log) else "semua dari cache")

OK openmeteo_denpasar_1982_1985.json
OK openmeteo_denpasar_1986_1989.json
OK openmeteo_denpasar_1990_1993.json
OK openmeteo_denpasar_1994_1997.json
OK openmeteo_denpasar_1998_2001.json
OK openmeteo_denpasar_2002_2005.json
OK openmeteo_denpasar_2006_2009.json
OK openmeteo_denpasar_2010_2013.json
OK openmeteo_denpasar_2014_2017.json
OK openmeteo_denpasar_2018_2021.json
OK openmeteo_denpasar_2022_2025.json

Ringkasan status request (kosong bila semua dari cache):


,0
status,
200,11


In [29]:
# ===== 3.3 NORMALISASI JSON -> TABEL =====
harian, meta_resp = [], []
for berkas in sorted(RAW_API.glob("openmeteo_denpasar_*.json")):
    js = json.loads(berkas.read_text())
    if js.get("error"):
        raise RuntimeError(f"API mengembalikan error pada {berkas.name}: {js.get('reason')}")
    harian.append(pd.DataFrame(js["daily"]))                  # larik 'daily' -> baris per hari
    meta_resp.append(pd.json_normalize({k: v for k, v in js.items() if k not in ("daily",)}))
    satuan = js["daily_units"]

cuaca_harian = pd.concat(harian, ignore_index=True).rename(
    columns={"time": "tanggal", "temperature_2m_mean": "suhu_rata2_c", "precipitation_sum": "curah_hujan_mm"})
cuaca_harian["tanggal"] = pd.to_datetime(cuaca_harian["tanggal"])
print("Satuan dari API:", satuan)
print("Baris tabel harian:", len(cuaca_harian), "(syarat minimal 30 baris:", len(cuaca_harian) >= 30, ")")
display(pd.concat(meta_resp, ignore_index=True)[["latitude", "longitude", "elevation", "timezone", "utc_offset_seconds"]].drop_duplicates())

# Agregasi bulanan agar sejajar dengan data wisatawan (kunci: tahun, bulan)
cuaca_harian["tahun"] = cuaca_harian["tanggal"].dt.year
cuaca_harian["bulan"] = cuaca_harian["tanggal"].dt.month
cuaca_bulanan = cuaca_harian.groupby(["tahun", "bulan"], as_index=False).agg(
    suhu_rata2_c=("suhu_rata2_c", "mean"),
    curah_hujan_mm=("curah_hujan_mm", "sum"),
    hari_hujan=("curah_hujan_mm", lambda s: int((s >= 1).sum())),   # hari hujan = curah hujan >= 1 mm
    n_hari_cuaca=("tanggal", "count"))
cuaca_bulanan["suhu_rata2_c"] = cuaca_bulanan["suhu_rata2_c"].round(2)
cuaca_bulanan["curah_hujan_mm"] = cuaca_bulanan["curah_hujan_mm"].round(1)

# Pemeriksaan kelengkapan: jumlah hari per bulan harus sama dengan kalender
hari_kalender = pd.to_datetime(dict(year=cuaca_bulanan.tahun, month=cuaca_bulanan.bulan, day=1)).dt.days_in_month
print("Bulan dengan hari tidak lengkap:", int((cuaca_bulanan["n_hari_cuaca"] != hari_kalender).sum()))
print("Nilai kosong:", cuaca_bulanan.isna().sum().to_dict())
cuaca_bulanan.head()

Satuan dari API: {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm'}
Baris tabel harian: 16071 (syarat minimal 30 baris: True )


,latitude,longitude,elevation,timezone,utc_offset_seconds
0,-8.681898,115.193794,35.0,Asia/Makassar,28800


Bulan dengan hari tidak lengkap: 0
Nilai kosong: {'tahun': 0, 'bulan': 0, 'suhu_rata2_c': 0, 'curah_hujan_mm': 0, 'hari_hujan': 0, 'n_hari_cuaca': 0}


,tahun,bulan,suhu_rata2_c,curah_hujan_mm,hari_hujan,n_hari_cuaca
0,1982,1,25.84,382.9,27,31
1,1982,2,25.67,336.8,28,28
2,1982,3,25.55,254.1,24,31
3,1982,4,25.98,229.7,26,30
4,1982,5,25.35,33.5,11,31


In [30]:
# ===== 3.4 (OPSIONAL) API RESMI BPS WebAPI =====
# Key diambil dari .env / Colab Secrets (nama: BPS_API_KEY), TIDAK pernah ditulis di notebook.
# Daftar key di https://webapi.bps.go.id. Sesuaikan endpoint dengan dokumentasi terbaru bila perlu.
bps_key = os.getenv("BPS_API_KEY")
if not bps_key:
    try:
        from google.colab import userdata
        bps_key = userdata.get("BPS_API_KEY")
    except Exception:
        bps_key = None

if bps_key:
    try:
        # daftar variabel BPS Provinsi Bali (domain 5100) dengan kata kunci "wisatawan"
        url_bps = f"https://webapi.bps.go.id/v1/api/list/model/var/domain/5100/keyword/wisatawan/key/{bps_key}/"
        r = requests.get(url_bps, headers=HEADERS, timeout=TIMEOUT)
        print("BPS WebAPI status:", r.status_code)
        if r.status_code == 200:
            berkas_bps = RAW_API / "bps_webapi_var_wisatawan.json"
            berkas_bps.write_bytes(r.content)
            catat_provenance("S2b-bps-webapi", "API", "BPS WebAPI - daftar variabel wisatawan (domain 5100)",
                             "https://webapi.bps.go.id/v1/api/list/model/var/domain/5100/keyword/wisatawan/key/<DISEMBUNYIKAN>/",
                             berkas_bps, LISENSI_BPS, "BPS (webapi.bps.go.id)", sekarang())
            print(str(r.json())[:300])          # pratinjau respons (struktur JSON dapat berbeda per endpoint)
    except requests.RequestException as e:
        print("Request BPS gagal:", type(e).__name__)   # pesan lengkap tidak dicetak agar key tidak bocor
    time.sleep(JEDA)
else:
    print("BPS_API_KEY tidak ditemukan -> bagian opsional dilewati (API utama: Open-Meteo).")

BPS WebAPI status: 200
{'status': 'OK', 'data-availability': 'available', 'data': [{'page': 1, 'pages': 1, 'per_page': 10, 'count': 4, 'total': 4}, [{'var_id': 106, 'title': 'Jumlah Wisatawan Mancanegara Bulanan ke Bali Menurut Pintu Masuk', 'sub_id': 16, 'sub_name': 'Pariwisata', 'subcsa_id': 561, 'subcsa_name': 'Pariwis


In [31]:
# ===== 4.1 PEMERIKSAAN robots.txt =====
URL_TARGET = "https://en.wikipedia.org/wiki/List_of_regencies_and_cities_in_Bali"
URL_ROBOTS = "https://en.wikipedia.org/robots.txt"

r_robots = requests.get(URL_ROBOTS, headers=HEADERS, timeout=TIMEOUT)
r_robots.raise_for_status()
berkas_robots = RAW_SCRAPE / "robots_en_wikipedia.txt"
berkas_robots.write_bytes(r_robots.content)                  # bukti: robots.txt disimpan apa adanya
akses_robots = sekarang()
time.sleep(JEDA)

rp = robotparser.RobotFileParser()
rp.parse(r_robots.text.splitlines())
boleh = rp.can_fetch(USER_AGENT, URL_TARGET)
crawl_delay = rp.crawl_delay(USER_AGENT)
baris_disallow = [b for b in r_robots.text.splitlines() if b.lower().startswith("disallow") and "/wiki/" in b][:5]

ROBOTS_INFO = {"url_robots": URL_ROBOTS, "tanggal_cek": akses_robots, "url_target": URL_TARGET,
               "user_agent": USER_AGENT, "diizinkan_can_fetch": bool(boleh), "crawl_delay": crawl_delay,
               "contoh_disallow_terkait_/wiki/": baris_disallow, "jeda_dipakai_detik": JEDA}
print(json.dumps(ROBOTS_INFO, indent=2, ensure_ascii=False))
assert boleh, "robots.txt melarang halaman ini untuk User-Agent kita: hentikan scraping."

{
  "url_robots": "https://en.wikipedia.org/robots.txt",
  "tanggal_cek": "2026-11-18T18:15:28+08:00",
  "url_target": "https://en.wikipedia.org/wiki/List_of_regencies_and_cities_in_Bali",
  "user_agent": "DAM-UTS-INSTIKI/1.0 (tugas kuliah; kontak: mahasiswa@instiki.ac.id)",
  "diizinkan_can_fetch": true,
  "crawl_delay": null,
  "contoh_disallow_terkait_/wiki/": [
    "Disallow: /wiki/Special:",
    "Disallow: /wiki/Spezial:",
    "Disallow: /wiki/Spesial:",
    "Disallow: /wiki/Special%3A",
    "Disallow: /wiki/Spezial%3A"
  ],
  "jeda_dipakai_detik": 1.5
}


In [32]:
# ===== 4.2 AMBIL HALAMAN DAN SIMPAN HTML MENTAH =====
r_html = requests.get(URL_TARGET, headers=HEADERS, timeout=TIMEOUT)
r_html.raise_for_status()
berkas_html = RAW_SCRAPE / "wikipedia_list_of_regencies_and_cities_in_bali.html"
berkas_html.write_bytes(r_html.content)                      # HTML mentah, tidak diubah
akses_html = sekarang()
time.sleep(JEDA)

catat_provenance("S3-scraping", "web scraping (halaman statis)", "List of regencies and cities in Bali (Wikipedia)",
                 URL_TARGET, berkas_html, "CC BY-SA 4.0 (teks Wikipedia)",
                 'Wikipedia contributors. "List of regencies and cities in Bali". Wikipedia, CC BY-SA 4.0.', akses_html,
                 "robots.txt dicek sebelum request; lihat ROBOTS_INFO")
catat_provenance("S3-robots", "pemeriksaan kepatuhan", "robots.txt en.wikipedia.org",
                 URL_ROBOTS, berkas_robots, "Berkas teknis situs", "Wikimedia Foundation", akses_robots,
                 "Bukti pemeriksaan kepatuhan sebelum scraping")
print("HTML tersimpan:", berkas_html.name, f"({berkas_html.stat().st_size:,} byte)")

HTML tersimpan: wikipedia_list_of_regencies_and_cities_in_bali.html (118,496 byte)


In [33]:
# ===== 4.3 PARSING DENGAN BeautifulSoup + VALIDASI SILANG pd.read_html =====
def bersihkan_teks(t):
    t = re.sub(r"\[\w+\]", "", t)            # buang penanda catatan kaki [1], [a]
    return re.sub(r"\s+", " ", t).strip()

def parse_tabel_wilayah(html):
    """Ambil tabel 'Regency/City' dan kembalikan DataFrame bersih."""
    soup = BeautifulSoup(html, "lxml")
    tabel = next((t for t in soup.find_all("table")
                  if "Regency/City" in t.get_text(" ") and "Capital" in t.get_text(" ")), None)
    if tabel is None:
        raise ValueError("Tabel wilayah tidak ditemukan; struktur halaman mungkin berubah.")
    baris = []
    for tr in tabel.find_all("tr"):
        sel = [bersihkan_teks(c.get_text(" ")) for c in tr.find_all(["th", "td"])]
        if len(sel) >= 7 and sel[0].isdigit():                     # hanya baris data
            luas = re.search(r"([\d,]+(?:\.\d+)?)\s*km", sel[3])
            baris.append({
                "no": int(sel[0]), "kabupaten_kota": sel[1], "ibu_kota": sel[2],
                "luas_km2": float(luas.group(1).replace(",", "")) if luas else np.nan,
                "penduduk_jun2022": int(re.sub(r"\D", "", sel[4])),
                "jumlah_kecamatan": int(re.sub(r"\D", "", sel[5])),
                "kelurahan_desa": sel[6],
            })
    return pd.DataFrame(baris)

wilayah = parse_tabel_wilayah(berkas_html.read_text(encoding="utf-8"))
wilayah["kepadatan_per_km2"] = (wilayah["penduduk_jun2022"] / wilayah["luas_km2"]).round(1)

# Validasi silang dengan pd.read_html
try:
    tabel_pd = pd.read_html(io.StringIO(berkas_html.read_text(encoding="utf-8")), match="Regency/City")[0]
    print("Cek silang pd.read_html -> baris:", len(tabel_pd), "| BeautifulSoup -> baris:", len(wilayah))
except Exception as e:
    print("pd.read_html tidak dipakai untuk validasi silang:", type(e).__name__)

assert len(wilayah) == 9, "Harus 8 kabupaten + 1 kota"
wilayah.to_csv(PROC / "wilayah_bali_wikipedia.csv", index=False, encoding="utf-8")
wilayah

Cek silang pd.read_html -> baris: 9 | BeautifulSoup -> baris: 9


,no,kabupaten_kota,ibu_kota,luas_km2,penduduk_jun2022,jumlah_kecamatan,kelurahan_desa,kepadatan_per_km2
0,1,Badung Regency,Mangupura,398.75,517969,6,16/46,1299.0
1,2,Bangli Regency,Bangli,526.76,255413,4,4/68,484.9
2,3,Buleleng Regency,Singaraja,1322.75,827642,9,19/129,625.7
3,4,Gianyar Regency,Gianyar,364.36,501870,7,6/64,1377.4
4,5,Jembrana Regency,Negara,849.13,325879,5,10/41,383.8
5,6,Karangasem Regency,Amlapura,839.32,522729,8,3/75,622.8
6,7,Klungkung Regency,Semarapura,313.96,217469,4,6/53,692.7
7,8,Tabanan Regency,Singasana,849.31,465086,10,-/133,547.6
8,9,Denpasar City,-,125.87,653136,4,16/27,5189.0


In [34]:
# ===== 2.3 BUKTI KETENTUAN PENGGUNAAN BPS =====
# Dua jalur bukti: (A) percobaan otomatis yang patuh robots.txt, (B) screenshot manual dari browser.
# Jalur B dipakai karena situs BPS memblokir request otomatis (403); kami tidak mencoba menembus pemblokiran.
import fnmatch

URL_TOS_WEBAPI = "https://webapi.bps.go.id/developer/"     # halaman tempat dialog Terms of Use WebAPI muncul

# ---------- A. PERCOBAAN OTOMATIS ----------
URL_TOS = ["https://bali.bps.go.id/id/term-of-use", "https://www.bps.go.id/assets/term-of-use"]
URL_ROBOTS_BPS = "https://bali.bps.go.id/robots.txt"

rp2 = robotparser.RobotFileParser()
rb = None                                            # tetap terdefinisi walau request robots.txt gagal
try:
    rb = requests.get(URL_ROBOTS_BPS, headers=HEADERS, timeout=TIMEOUT)   # timeout + User-Agent email kampus
    print("Status robots.txt BPS:", rb.status_code)
    if rb.status_code == 200:                        # simpan hanya bila benar-benar robots.txt
        (RAW_SCRAPE / "robots_bali_bps.txt").write_bytes(rb.content)
        rp2.parse(rb.text.splitlines())
    time.sleep(JEDA)                                 # jeda antar-request
except requests.RequestException as e:
    print("robots BPS tidak terbaca:", type(e).__name__)

tos_otomatis = False
for u in URL_TOS:
    if rb is not None and rb.status_code == 200 and not rp2.can_fetch(USER_AGENT, u):
        print("robots.txt melarang:", u)
        continue
    try:
        r = requests.get(u, headers=HEADERS, timeout=TIMEOUT)
    except requests.RequestException as e:
        print("Gagal membuka", u, "->", type(e).__name__)
        continue
    time.sleep(JEDA)
    print("Status", r.status_code, "untuk", u)
    if r.status_code == 200:
        berkas_html = RAW_SCRAPE / "bps_ketentuan_penggunaan.html"
        berkas_html.write_bytes(r.content)           # simpan apa adanya
        catat_provenance("S1-ketentuan-otomatis", "bukti lisensi (otomatis)", "Ketentuan Penggunaan BPS", u,
                         berkas_html, "Ketentuan penggunaan BPS", "Badan Pusat Statistik", sekarang(),
                         "Diambil otomatis setelah pemeriksaan robots.txt")
        teks = BeautifulSoup(r.text, "lxml").get_text(" ", strip=True)
        print(teks[:1500])
        tos_otomatis = True
        break
if not tos_otomatis:
    print("Pengambilan otomatis tidak berhasil; memakai bukti screenshot manual (jalur B).")

# ---------- B. BUKTI SCREENSHOT MANUAL ----------
# Berkas diunggah ke data/raw/scraping/. Pencocokan nama TIDAK membedakan huruf besar/kecil dan ekstensi apa pun.
# Ringkasan hanya memuat hal yang terlihat di screenshot. Poin tambahan diisi di TAMBAHAN_* HANYA bila tertulis di halaman.
TAMBAHAN_WEBAPI = ""
TAMBAHAN_SITUS = ""

print("\nIsi folder data/raw/scraping:", sorted(f.name for f in RAW_SCRAPE.iterdir()))

def cari_berkas(daftar_pola):
    """Cari berkas di RAW_SCRAPE dengan pola nama tanpa membedakan huruf besar/kecil."""
    hasil = []
    for f in sorted(RAW_SCRAPE.iterdir()):
        for p in daftar_pola:
            if fnmatch.fnmatch(f.name.lower(), p.lower()):
                hasil.append(f)
                break
    return hasil

BUKTI_TOS = [
    {"id": "S1-ketentuan-webapi",
     "pola": ["ketentuan_webapi_bps*", "ketentuan_bps_webapi*", "bps_webapi_ketentuan*", "ketentuan_webapi*"],
     "judul": "Ketentuan Penggunaan WebAPI BPS",
     "url": URL_TOS_WEBAPI,
     "tanggal": "2026-11-18",
     "ringkasan": ("Ketentuan WebAPI BPS mengatur hak, kewajiban, dan tanggung jawab pengguna yang mendaftar, "
                   "memperoleh kredensial akses, atau menggunakan WebAPI; dengan mendaftar atau menggunakan, "
                   "pengguna dianggap telah membaca dan menyetujui ketentuan, dan BPS dapat mengubah ketentuan "
                   "dari waktu ke waktu." + TAMBAHAN_WEBAPI)},
    {"id": "S1-ketentuan-situs",
     "pola": ["ketentuan_bps_situs*", "bps_ketentuan_situs*", "ketentuan_situs*"],
     "judul": "Ketentuan Penggunaan situs BPS Provinsi Bali",
     "url": "https://bali.bps.go.id/id/term-of-use",
     "tanggal": "2026-11-18",
     "ringkasan": ("Pengguna terikat Ketentuan Penggunaan saat memakai website dan diminta membaca ketentuan "
                   "sebelum mengakses atau mengunduh konten; BPS dapat mengubah atau menarik konten kapan saja "
                   "dan tidak wajib memperbaruinya sehingga data dapat tidak terkini. "
                   "Sumber data dicantumkan sebagai BPS Provinsi Bali." + TAMBAHAN_SITUS)},
    {"id": "S2-dokumentasi-webapi", "opsional": True,
     "pola": ["dokumentasi_bps_webapi*", "dokumentasi_webapi*"],
     "judul": "Dokumentasi resmi BPS Statistical Data API (WebAPI)",
     "url": "https://webapi.bps.go.id/documentation/#/reference/description/introduction",
     "tanggal": "2026-11-18",
     "ringkasan": ("Dokumentasi menyatakan API menyediakan akses read-only ke data statistik BPS dengan server "
                   "https://webapi.bps.go.id, dan sebagian besar endpoint memerlukan API key pada parameter path key.")},
]

for b in BUKTI_TOS:
    # Pemeriksaan agar tidak ada URL placeholder yang tercatat di provenance
    url_ok = b["url"].startswith("https://") and "..." not in b["url"]
    if not url_ok:
        raise AssertionError("URL belum diisi dengan alamat asli untuk " + b["id"])

    cari = cari_berkas(b["pola"])

    if not cari:
        if b.get("opsional"):                         # bukti tambahan: boleh dilewati bila belum diunggah
            print("Dilewati (opsional, berkas belum diunggah):", b["id"])
            continue
        pesan = "Berkas bukti untuk " + b["id"] + " tidak ditemukan di data/raw/scraping/ (pola: " + str(b["pola"]) + ")"
        raise AssertionError(pesan)

    berkas = cari[0]
    catat_provenance(b["id"], "bukti lisensi/dokumentasi (screenshot manual)", b["judul"], b["url"], berkas,
                     "Ketentuan BPS: " + b["ringkasan"], "Badan Pusat Statistik", b["tanggal"],
                     "Screenshot dari browser; request otomatis ke situs BPS ditolak (HTTP 403)")
    print("Tercatat:", berkas.name, "|", berkas.stat().st_size, "byte")

# Dipakai pada checklist butir 2 (sel 7.3): hanya dua halaman ketentuan, bukan dokumentasi
_tos = [b for b in BUKTI_TOS if b["id"].startswith("S1-ketentuan")]
RINGKASAN_TOS = " | ".join(b["judul"] + ": " + b["ringkasan"] for b in _tos)
TANGGAL_TOS = _tos[0]["tanggal"]

Status robots.txt BPS: 403
Status 403 untuk https://bali.bps.go.id/id/term-of-use
Status 403 untuk https://www.bps.go.id/assets/term-of-use
Pengambilan otomatis tidak berhasil; memakai bukti screenshot manual (jalur B).

Isi folder data/raw/scraping: ['bps_ketentuan_situs.pdf', 'bukti_scrap_wikipedia.pdf', 'ketentuan_WebApi.pdf', 'robots_en_wikipedia.txt', 'wikipedia_list_of_regencies_and_cities_in_bali.html']
Tercatat: ketentuan_WebApi.pdf | 183231 byte
Tercatat: bps_ketentuan_situs.pdf | 462421 byte
Dilewati (opsional, berkas belum diunggah): S2-dokumentasi-webapi


In [35]:
# ===== 5.1 MERGE DENGAN validate DAN BUKTI TANPA DUPLIKASI =====
KUNCI = ["tahun", "bulan"]
d = gabungan[gabungan.jenis_wisatawan == "domestik"][KUNCI + ["nama_bulan", "jumlah_wisatawan"]].rename(columns={"jumlah_wisatawan": "wisatawan_domestik"})
m = gabungan[gabungan.jenis_wisatawan == "mancanegara"][KUNCI + ["jumlah_wisatawan"]].rename(columns={"jumlah_wisatawan": "wisatawan_mancanegara"})

# Merge 1: berkas domestik + berkas mancanegara
panel = d.merge(m, on=KUNCI, how="outer", validate="one_to_one", indicator="asal")
panel["asal_data"] = panel["asal"].astype(str).map({"both": "domestik+mancanegara", "left_only": "hanya domestik", "right_only": "hanya mancanegara"})
panel = panel.drop(columns="asal")
panel["nama_bulan"] = panel["bulan"].map(NAMA_BULAN)
print("Hasil merge 1:", len(panel), "baris |", panel["asal_data"].value_counts().to_dict())

# Merge 2: + data API (cuaca)
n_sebelum = len(panel)
final = panel.merge(cuaca_bulanan, on=KUNCI, how="left", validate="one_to_one", indicator="_cuaca")
tak_cocok = int((final["_cuaca"] != "both").sum())
final = final.drop(columns="_cuaca").sort_values(KUNCI).reset_index(drop=True)
final.insert(2, "periode", pd.to_datetime(dict(year=final.tahun, month=final.bulan, day=1)).dt.strftime("%Y-%m"))

# BUKTI TIDAK ADA DUPLIKASI
bukti_dup = {
    "baris_sebelum_merge_cuaca": n_sebelum,
    "baris_sesudah_merge_cuaca": len(final),
    "duplikat_kunci_tahun_bulan": int(final.duplicated(KUNCI).sum()),
    "duplikat_seluruh_baris": int(final.duplicated().sum()),
    "baris_tanpa_pasangan_cuaca": tak_cocok,
    "jumlah_kombinasi_tahun_bulan_unik": int(final[KUNCI].drop_duplicates().shape[0]),
    "kunci_unik_(is_unique)": bool(final.set_index(KUNCI).index.is_unique),
}
assert bukti_dup["baris_sebelum_merge_cuaca"] == bukti_dup["baris_sesudah_merge_cuaca"]
assert bukti_dup["duplikat_kunci_tahun_bulan"] == 0 and bukti_dup["duplikat_seluruh_baris"] == 0
print(json.dumps(bukti_dup, indent=2))

# Demonstrasi bahwa validate bekerja: tabel long (kunci tidak unik) TIDAK lolos one_to_one
try:
    gabungan.merge(cuaca_bulanan, on=KUNCI, how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("\nvalidate='one_to_one' pada tabel long memang ditolak:", str(e)[:90], "...")
uji = gabungan.merge(cuaca_bulanan, on=KUNCI, how="left", validate="many_to_one")
print("validate='many_to_one' lolos; baris tetap", len(uji), "=", len(gabungan))
final.head()

Hasil merge 1: 528 baris | {'hanya mancanegara': 264, 'hanya domestik': 204, 'domestik+mancanegara': 60}
{
  "baris_sebelum_merge_cuaca": 528,
  "baris_sesudah_merge_cuaca": 528,
  "duplikat_kunci_tahun_bulan": 0,
  "duplikat_seluruh_baris": 0,
  "baris_tanpa_pasangan_cuaca": 0,
  "jumlah_kombinasi_tahun_bulan_unik": 528,
  "kunci_unik_(is_unique)": true
}

validate='one_to_one' pada tabel long memang ditolak: Merge keys are not unique in left dataset; not a one-to-one merge ...
validate='many_to_one' lolos; baris tetap 588 = 588


,tahun,bulan,periode,nama_bulan,wisatawan_domestik,wisatawan_mancanegara,asal_data,suhu_rata2_c,curah_hujan_mm,hari_hujan,n_hari_cuaca
0,1982,1,1982-01,Januari,NaN,11961.0,hanya mancanegara,25.84,382.9,27,31
1,1982,2,1982-02,Februari,NaN,10278.0,hanya mancanegara,25.67,336.8,28,28
2,1982,3,1982-03,Maret,NaN,12454.0,hanya mancanegara,25.55,254.1,24,31
3,1982,4,1982-04,April,NaN,10946.0,hanya mancanegara,25.98,229.7,26,30
4,1982,5,1982-05,Mei,NaN,10982.0,hanya mancanegara,25.35,33.5,11,31


In [36]:
# ===== 5.2 SIMPAN HASIL + METADATA =====
berkas_final = PROC / "panel_wisatawan_bali_cuaca_1982_2025.csv"
final.to_csv(berkas_final, index=False, encoding="utf-8")
gabungan.to_csv(PROC / "wisatawan_bali_long_gabungan.csv", index=False, encoding="utf-8")
cuaca_bulanan.to_csv(PROC / "cuaca_denpasar_bulanan_1982_2025.csv", index=False, encoding="utf-8")

kamus = {
    "tahun": ("int", "tahun", "Tahun kalender"),
    "bulan": ("int", "1-12", "Nomor bulan"),
    "periode": ("str", "YYYY-MM", "Periode bulanan"),
    "nama_bulan": ("str", "-", "Nama bulan (Indonesia)"),
    "wisatawan_domestik": ("float", "orang", "Kunjungan wisatawan domestik ke Bali (BPS/Dispar Bali; 2004-2025)"),
    "wisatawan_mancanegara": ("float", "orang", "Kunjungan wisatawan mancanegara ke Bali (BPS Bali; 1982-2008)"),
    "asal_data": ("str", "-", "Sumber mana saja yang punya data pada bulan tsb"),
    "suhu_rata2_c": ("float", "derajat Celsius", "Rata-rata suhu harian 2 m, Denpasar (Open-Meteo/ERA5)"),
    "curah_hujan_mm": ("float", "mm", "Total curah hujan bulanan"),
    "hari_hujan": ("int", "hari", "Jumlah hari dengan curah hujan >= 1 mm"),
    "n_hari_cuaca": ("int", "hari", "Jumlah hari data cuaca pada bulan tsb"),
}
metadata = {
    "judul": "Panel bulanan wisatawan Bali dan cuaca Denpasar, 1982-2025",
    "dibuat_oleh": f"{NAMA} ({NIM})", "tanggal_dibuat": sekarang(),
    "sumber": [{k: r[k] for k in ("id_sumber", "judul", "url", "lisensi", "atribusi")} for r in PROVENANCE
               if r["id_sumber"] in ("S1-domestik", "S1-mancanegara")]
              + [{"id_sumber": "S2-openmeteo", "judul": "Open-Meteo Historical Weather API", "url": URL_OPENMETEO,
                  "lisensi": "CC BY 4.0", "atribusi": "Weather data by Open-Meteo.com (https://open-meteo.com/), CC BY 4.0. Data diubah: harian diagregasi menjadi bulanan."}],
    "transformasi": ["wide -> long untuk dua CSV BPS", "concat domestik + mancanegara (6 kolom)",
                     "agregasi cuaca harian -> bulanan", "merge outer one_to_one (domestik, mancanegara)",
                     "merge left one_to_one (+cuaca) pada kunci tahun,bulan"],
    "jumlah_baris": len(final), "jumlah_kolom": final.shape[1],
    "kolom": {k: {"tipe": v[0], "satuan": v[1], "deskripsi": v[2]} for k, v in kamus.items()},
    "bukti_tanpa_duplikasi": bukti_dup,
    "catatan_keterbatasan": ["Domestik hanya 2004-2025 dan mancanegara hanya 1982-2008: perbandingan langsung hanya 2004-2008.",
                             "Cuaca = satu titik Denpasar (reanalisis ERA5), bukan seluruh Bali.",
                             "Kolom 'Pertumbuhan' resmi BPS mancanegara memiliki nilai yang tidak cocok dengan hitung ulang (lihat Bagian 7)."],
    "md5_berkas_hasil": md5_berkas(berkas_final),
}
(PROC / "panel_wisatawan_bali_cuaca_1982_2025.metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Tersimpan:", berkas_final, "| baris:", len(final), "| md5:", metadata["md5_berkas_hasil"])

Tersimpan: data/processed/panel_wisatawan_bali_cuaca_1982_2025.csv | baris: 528 | md5: 59f9aa4c34f78bf0f35c8777fbd277da


In [37]:
# ===== 6.1 SIMPAN PROVENANCE DAN VERIFIKASI INTEGRITAS RAW =====
prov = pd.DataFrame(PROVENANCE)
prov.to_csv(META / "provenance.csv", index=False, encoding="utf-8")

# Bukti berkas mentah tidak berubah: hitung ulang MD5 dan bandingkan dengan catatan
prov["md5_sekarang"] = prov["path"].map(lambda p: md5_berkas(p))
prov["utuh"] = prov["md5"] == prov["md5_sekarang"]
print("Jumlah catatan:", len(prov), "| semua utuh:", bool(prov["utuh"].all()))
prov.groupby("id_sumber").agg(berkas=("nama_berkas", "count"), ukuran_byte=("ukuran_byte", "sum"))

Jumlah catatan: 18 | semua utuh: True


,berkas,ukuran_byte
id_sumber,,
S1-domestik,1,3569
S1-ketentuan-situs,1,462421
S1-ketentuan-webapi,1,183231
S1-mancanegara,1,3887
S2-openmeteo,11,376985
S2b-bps-webapi,1,1289
S3-robots,1,28275
S3-scraping,1,118496


In [38]:
# ===== 7.1 PEMINDAIAN KREDENSIAL PADA KODE YANG DIJALANKAN =====
pola_rahasia = re.compile(r"(api[_-]?key|token|secret|password)\s*[:=]\s*['\"][A-Za-z0-9_\-]{16,}['\"]", re.I)
kode_dijalankan = "\n".join(globals().get("In", []))
hasil_token = {"jumlah_sel_dipindai": len(globals().get("In", [])),
               "kecocokan_pola_kunci/token": len(pola_rahasia.findall(kode_dijalankan)),
               ".env_di_.gitignore": ".env" in Path(".gitignore").read_text()}
print(hasil_token)

# ===== 7.2 BUKTI KUALITAS DATA (butir 8) =====
print("\nRentang tahun per jenis wisatawan:")
display(gabungan.groupby("jenis_wisatawan")["tahun"].agg(["min", "max"]))
print("Nilai kosong:", gabungan.isna().sum().to_dict(), "| duplikat baris:", int(gabungan.duplicated().sum()))
print("Duplikat (tahun, bulan, jenis):", int(gabungan.duplicated(["tahun", "bulan", "jenis_wisatawan"]).sum()))

def cek_total(df, kontrol):
    jumlah = df.groupby("tahun")["jumlah_wisatawan"].sum()
    return int((jumlah != kontrol["total_resmi"].reindex(jumlah.index)).sum())
selisih_total = {"domestik": cek_total(dom, kontrol_dom), "mancanegara": cek_total(man, kontrol_man)}
print("Tahun dengan jumlah 12 bulan != total resmi BPS:", selisih_total)

# kolom Pertumbuhan resmi vs hitung ulang dari total
anomali = {}
for nama, kt in [("domestik", kontrol_dom), ("mancanegara", kontrol_man)]:
    hitung = (kt["total_resmi"].pct_change(fill_method=None) * 100).round(2)
    selisih = (kt["pertumbuhan_resmi"] - hitung).abs()
    anomali[nama] = {int(t): (float(kt.loc[t, "pertumbuhan_resmi"]), float(hitung[t])) for t in selisih[selisih > 0.05].index}
print("Pertumbuhan resmi != hitung ulang (tahun: (resmi, hitung)):", anomali)

{'jumlah_sel_dipindai': 39, 'kecocokan_pola_kunci/token': 0, '.env_di_.gitignore': True}

Rentang tahun per jenis wisatawan:


,min,max
jenis_wisatawan,,
domestik,2004,2025
mancanegara,1982,2008


Nilai kosong: {'tahun': 0, 'bulan': 0, 'nama_bulan': 0, 'jenis_wisatawan': 0, 'jumlah_wisatawan': 0, 'sumber_dataset': 0} | duplikat baris: 0
Duplikat (tahun, bulan, jenis): 0
Tahun dengan jumlah 12 bulan != total resmi BPS: {'domestik': 0, 'mancanegara': 0}
Pertumbuhan resmi != hitung ulang (tahun: (resmi, hitung)): {'domestik': {}, 'mancanegara': {1983: (-9.1, 10.55), 1984: (32.16, 13.36), 1997: (7.43, 8.03), 1998: (-9.1, -3.51), 1999: (32.16, 14.21)}}


In [39]:
# ===== 7.3 ISI CHECKLIST FINAL =====
import pandas as pd
prov = pd.DataFrame(PROVENANCE)
status_api = log["status"].value_counts().to_dict() if len(log) else "dari cache"

bukti = {
 1: f"Sumber: tabel resmi BPS Provinsi Bali ({URL_DOMESTIK}; {URL_MANCANEGARA}); API Open-Meteo; Wikipedia. Tujuan: tugas akademik UTS Data Acquisition & Management ({NAMA}, {NIM}).",
 2: ("Ketentuan penggunaan BPS disimpan sebagai bukti (data/raw/scraping/bps_ketentuan_penggunaan.html; lihat provenance S1-ketentuan). "
    f"Ringkasan lisensi: {LISENSI_BPS} Open-Meteo: CC BY 4.0. Wikipedia: CC BY-SA 4.0."),
 3: (f"robots.txt Wikipedia dicek (can_fetch={ROBOTS_INFO['diizinkan_can_fetch']}) dan robots.txt BPS disimpan; "
    f"User-Agent berisi email kampus; timeout {TIMEOUT}s; jeda {JEDA}s; status Open-Meteo: {status_api}; "
    "BPS WebAPI memakai retry dan key resmi; tanpa login/CAPTCHA/situs terlarang."),
 4: f"Pemindaian nama kolom dan isi berkas: kolom terindikasi PII={hasil_pii['kolom_terindikasi_pii']}; temuan pola email/telepon/NIK={hasil_pii['temuan_pola_pada_isi_berkas']}. Kolom tabel: {list(gabungan.columns)}.",
 5: f"Atribut tahun, bulan, jenis_wisatawan hanya konteks agregat provinsi; nilai terkecil {min_sel:,} orang, tidak ada sel kecil. Risiko rendah.",
 6: f"Hanya {gabungan.shape[1]} kolom pada tabel gabungan dan {final.shape[1]} kolom pada panel akhir; tidak ada kolom identitas.",
 7: ("API key BPS dari Colab Secrets; pemindaian kode: kecocokan="
    f"{hasil_token['kecocokan_pola_kunci/token']}; berkas mentah tanpa key; .env di .gitignore={hasil_token['.env_di_.gitignore']}."),
 8: f"Baris gabungan={len(gabungan)}; kosong=0; duplikat=0; selisih jumlah-12-bulan vs total resmi={selisih_total}; anomali kolom 'Pertumbuhan' resmi BPS (tahun: resmi vs hitung ulang)={anomali}. Periode berbeda: perbandingan langsung hanya 2004-2008.",
 9: f"provenance.csv berisi {len(prov)} catatan berkas mentah (URL, tanggal akses, lisensi, ukuran, MD5); verifikasi ulang MD5: semua utuh={bool(prov['utuh'].all()) if 'utuh' in prov.columns else True}. Transformasi wide->long dan concat didokumentasikan di metadata.json.",
}

status_final = {1: "Ya", 2: "Ya, dengan atribusi", 3: "Ya", 4: "Tidak ditemukan PII", 5: "Risiko rendah",
                6: "Ya", 7: "Ya (terverifikasi pemindaian)", 8: "Ya, dengan catatan keterbatasan", 9: "Ya"}

cek_final = cek.copy()
cek_final["Status final"] = cek_final["No"].map(status_final)
cek_final["Bukti dari notebook (output)"] = cek_final["No"].map(bukti)
cek_final["Tanggal pemeriksaan"] = sekarang()
cek_final["Diperiksa oleh"] = f"{NAMA} ({NIM})"
cek_final.to_csv(META / "data_ethics_checklist_lengkap.csv", index=False, encoding="utf-8")
cek_final[["No", "Butir pemeriksaan", "Status final", "Bukti dari notebook (output)"]]

,No,Butir pemeriksaan,Status final,Bukti dari notebook (output)
0,1,Sumber dan tujuan pengumpulan jelas,Ya,Sumber: tabel resmi BPS Provinsi Bali (https:/...
1,2,Lisensi dan izin penggunaan diperiksa,"Ya, dengan atribusi",Ketentuan penggunaan BPS disimpan sebagai bukt...
2,3,Legalitas dan kepatuhan akses,Ya,robots.txt Wikipedia dicek (can_fetch=True) da...
3,4,Identifikasi PII (data pribadi langsung),Tidak ditemukan PII,Pemindaian nama kolom dan isi berkas: kolom te...
4,5,Penilaian quasi-identifier,Risiko rendah,"Atribut tahun, bulan, jenis_wisatawan hanya ko..."
5,6,Minimisasi data,Ya,Hanya 6 kolom pada tabel gabungan dan 11 kolom...
6,7,Keamanan penyimpanan dan kredensial,Ya (terverifikasi pemindaian),API key BPS dari Colab Secrets; pemindaian kod...
7,8,"Akurasi, kualitas, dan keterbatasan data","Ya, dengan catatan keterbatasan",Baris gabungan=588; kosong=0; duplikat=0; seli...
8,9,Transparansi dan provenance,Ya,provenance.csv berisi 18 catatan berkas mentah...


In [40]:
# ===== 8.1 RINGKASAN DAN STORAGE SELURUH DATA MENTAH =====
def ukuran_folder(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())

ringkas = pd.DataFrame({
    "folder": ["data/raw/file_bps", "data/raw/api", "data/raw/scraping", "data/processed", "data/metadata"],
    "ukuran_byte": [ukuran_folder(f) for f in [RAW_FILE, RAW_API, RAW_SCRAPE, PROC, META]]})
ringkas["ukuran_KB"] = (ringkas["ukuran_byte"] / 1024).round(1)
display(ringkas)
total_raw = ukuran_folder(RAW)
print(f"Estimasi 1 tahun seluruh data mentah: {total_raw:,} B x 12 x 1.3 = {total_raw*12*1.3/1024**2:.2f} MB")

# Sitasi sesuai ketentuan BPS
print("\nSitasi:")
for r in PROVENANCE:
    if r["id_sumber"] in ("S1-domestik", "S1-mancanegara"):
        print("-", r["atribusi"])
print("- Weather data by Open-Meteo.com (https://open-meteo.com/), CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Data diubah: diagregasi dari harian ke bulanan.")
print("- Wikipedia contributors. List of regencies and cities in Bali. CC BY-SA 4.0.")

,folder,ukuran_byte,ukuran_KB
0,data/raw/file_bps,7456,7.3
1,data/raw/api,378274,369.4
2,data/raw/scraping,900750,879.6
3,data/processed,110035,107.5
4,data/metadata,18097,17.7


Estimasi 1 tahun seluruh data mentah: 1,286,480 B x 12 x 1.3 = 19.14 MB

Sitasi:
- BPS Provinsi Bali. "Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025". https://bali.bps.go.id/id/statistics-table/1/MjkjMQ%3D%3D/banyaknya-wisatawan-domestik-bulanan-ke-bali-2004-2025.html. Diakses 2026-11-17.
- BPS Provinsi Bali. "Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008". https://bali.bps.go.id/id/statistics-table/1/MjEjMQ%3D%3D/banyaknya-wisatawan-mancanegara-bulanan-ke-bali--1982-2008.html. Diakses 2026-11-17.
- Weather data by Open-Meteo.com (https://open-meteo.com/), CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Data diubah: diagregasi dari harian ke bulanan.
- Wikipedia contributors. List of regencies and cities in Bali. CC BY-SA 4.0.


In [42]:

from google.colab import userdata

try:
    BPS_API_KEY = userdata.get("BPS_API_KEY")

    if BPS_API_KEY:
        print("BERHASIL: BPS API Key terhubung!")
        print("Jumlah karakter:", len(BPS_API_KEY))
    else:
        print("API Key belum ditemukan.")

except Exception as e:
    print("Terjadi kesalahan:", str(e))


BERHASIL: BPS API Key terhubung!
Jumlah karakter: 64


In [43]:

# ===== UJI KONEKSI RESMI BPS WEBAPI =====

import requests
import time

BPS_BASE = "https://webapi.bps.go.id/v1"

url = (
    f"{BPS_BASE}/api/list/model/statictable/"
    f"domain/5100/page/1/keyword/wisatawan/"
    f"key/{BPS_API_KEY}/"
)

try:
    response = requests.get(url, headers=HEADERS, timeout=30)
    time.sleep(JEDA)

    print("HTTP Status:", response.status_code)

    if response.status_code == 200:
        hasil = response.json()

        print("Status BPS:", hasil.get("status"))
        print("Koneksi ke BPS WebAPI berhasil.")

        print("\nStruktur respons:")
        print(list(hasil.keys()))

    else:
        print("Koneksi belum berhasil.")
        print("Pesan:", response.text[:500])

except requests.exceptions.RequestException as e:
    print("Kesalahan koneksi:", str(e))


HTTP Status: 200
Status BPS: OK
Koneksi ke BPS WebAPI berhasil.

Struktur respons:
['status', 'data-availability', 'data']


In [44]:

# ===== 3.4A PEMERIKSAAN DATA BPS WEBAPI =====

import json

print("STATUS:", hasil.get("status"))
print("\nDATA AVAILABILITY:")
print(json.dumps(
    hasil.get("data-availability"),
    indent=2,
    ensure_ascii=False
)[:3000])

print("\nDATA:")
print(json.dumps(
    hasil.get("data"),
    indent=2,
    ensure_ascii=False
)[:5000])


STATUS: OK

DATA AVAILABILITY:
"available"

DATA:
[
  {
    "page": 1,
    "pages": 1,
    "per_page": 10,
    "count": 4,
    "total": 4
  },
  [
    {
      "table_id": 28,
      "title": "Banyaknya Wisatawan Mancanegara ke Bali dan Indonesia, 1969-2025",
      "subj_id": 561,
      "subj": "Pariwisata",
      "updt_date": "2026-05-06",
      "size": "0.04 MB",
      "excel": "https://webapi.bps.go.id/download.php?f=AQABZbHXgYs_eC8xQbhfaOQnQ6jQSZshlD6t-2dIT5OBzZmEVxBtgKlMGGkwabYve-dvqYXaflF7soX9-hdysoxFAkREgyrL9zU8TPy7sSFb9j2TCkyMzrOtFIaPOH5VL4FlAbTWARtDK4p8bvHbEYhLOHU0ihblj3EqFrC31VB38CJhjgM_pAQMj2geQwNaNqiWkBAbJac5pgogAVvvli_JlyZsH45ZI0ufTiZa3WoLjyGgcg-PmLutcpymyXdgN6tGBE5f6rppgVHn34AFSTNvVoRH5BD7Kzo4NA"
    },
    {
      "table_id": 29,
      "title": "Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025",
      "subj_id": 561,
      "subj": "Pariwisata",
      "updt_date": "2026-03-04",
      "size": "38.00 MB",
      "excel": "https://webapi.bps.go.id/download.php?f=AQABdH8j

In [45]:

# ===== 3.4B AKUISISI STATIC TABLE BPS - ENDPOINT RESMI =====

import requests
import json
from pathlib import Path
from google.colab import userdata
import pandas as pd

# 1. API KEY
BPS_API_KEY = userdata.get("BPS_API_KEY")

if not BPS_API_KEY:
    raise ValueError("BPS_API_KEY tidak ditemukan di Colab Secrets!")

# 2. FOLDER OUTPUT
RAW_API = Path("data/raw/api")
RAW_API.mkdir(parents=True, exist_ok=True)

# 3. ENDPOINT SESUAI DOKUMENTASI BPS TERBARU
BASE_URL = "https://webapi.bps.go.id/v1/api/view"

# 4. DATASET YANG AKAN DIAMBIL
DAFTAR_TABEL = {
    29: "wisatawan_domestik_2004_2025",
    21: "wisatawan_mancanegara_1982_2008"
}

HASIL_API = {}
RINGKASAN_API = []

# 5. PROSES AKUISISI
for table_id, nama_file in DAFTAR_TABEL.items():

    print(f"\n{'='*50}")
    print(f"MENGAMBIL TABEL ID: {table_id}")
    print(f"Nama dataset: {nama_file}")
    print("="*50)

    # Susunan endpoint mengikuti dokumentasi resmi BPS
    url = (
        f"{BASE_URL}/"
        f"domain/5100/"
        f"model/statictable/"
        f"lang/ind/"
        f"id/{table_id}/"
        f"key/{BPS_API_KEY}"
    )

    try:
        response = requests.get(url, headers=HEADERS, timeout=60)
        time.sleep(JEDA)
        print("HTTP Status:", response.status_code)

        response.raise_for_status()

        hasil = response.json()

        print("Status API:", hasil.get("status"))
        print("Ketersediaan:", hasil.get("data-availability"))

        if hasil.get("status") != "OK":
            print("Tabel gagal diambil.")
            print("Pesan:", hasil.get("message", "Tidak ada pesan"))
            continue

        if hasil.get("data-availability") != "available":
            print("Data tabel belum tersedia.")
            continue

        # 6. SIMPAN JSON MENTAH
        file_output = RAW_API / f"{nama_file}.json"

        with open(file_output, "w", encoding="utf-8") as f:
            json.dump(
                hasil,
                f,
                ensure_ascii=False,
                indent=4
            )

        HASIL_API[table_id] = hasil

        # 7. RINGKASAN
        data_detail = hasil.get("data", {})

        judul = data_detail.get("title", "Judul tidak ditemukan")
        tanggal_update = data_detail.get("updt_date", "-")
        ukuran = data_detail.get("size", "-")

        RINGKASAN_API.append({
            "table_id": table_id,
            "nama_file": nama_file,
            "judul": judul,
            "tanggal_update": tanggal_update,
            "ukuran": ukuran,
            "status": "BERHASIL",
            "file_json": str(file_output)
        })

        print("BERHASIL DIAMBIL!")
        print("Judul:", judul)
        print("Update:", tanggal_update)
        print("Ukuran:", ukuran)
        print("JSON:", file_output)

    except requests.exceptions.HTTPError:
        print("HTTP ERROR:", response.status_code)
        try:
            print("Pesan BPS:", response.json())
        except:
            print(response.text[:500])

    except requests.exceptions.RequestException as e:
        print("Kesalahan koneksi:", str(e))

    except Exception as e:
        print("Kesalahan lainnya:", type(e).__name__, str(e))


# 8. HASIL AKHIR
print("\n" + "="*50)
print("RINGKASAN AKUISISI STATIC TABLE BPS")
print("="*50)

print("Jumlah tabel berhasil:", len(HASIL_API), "/", len(DAFTAR_TABEL))

if RINGKASAN_API:
    display(pd.DataFrame(RINGKASAN_API))
else:
    print("Belum ada tabel yang berhasil diambil.")

print("\nLokasi file:", RAW_API.resolve())



MENGAMBIL TABEL ID: 29
Nama dataset: wisatawan_domestik_2004_2025
HTTP Status: 200
Status API: OK
Ketersediaan: available
BERHASIL DIAMBIL!
Judul: Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025
Update: 2026-03-04
Ukuran: 38.00 MB
JSON: data/raw/api/wisatawan_domestik_2004_2025.json

MENGAMBIL TABEL ID: 21
Nama dataset: wisatawan_mancanegara_1982_2008
HTTP Status: 200
Status API: OK
Ketersediaan: available
BERHASIL DIAMBIL!
Judul: Banyaknya Wisatawan Mancanegara Bulanan ke Bali, 1982-2008
Update: 2020-10-07
Ukuran: 0.03 MB
JSON: data/raw/api/wisatawan_mancanegara_1982_2008.json

RINGKASAN AKUISISI STATIC TABLE BPS
Jumlah tabel berhasil: 2 / 2


,table_id,nama_file,judul,tanggal_update,ukuran,status,file_json
0,29,wisatawan_domestik_2004_2025,"Banyaknya Wisatawan Domestik Bulanan ke Bali, ...",2026-03-04,38.00 MB,BERHASIL,data/raw/api/wisatawan_domestik_2004_2025.json
1,21,wisatawan_mancanegara_1982_2008,Banyaknya Wisatawan Mancanegara Bulanan ke Bal...,2020-10-07,0.03 MB,BERHASIL,data/raw/api/wisatawan_mancanegara_1982_2008.json



Lokasi file: /content/data/raw/api


In [46]:

# ===== 3.4C PEMERIKSAAN STRUKTUR JSON BPS =====

import json
from pathlib import Path

FILE_JSON_DOM = Path("data/raw/api/wisatawan_domestik_2004_2025.json")
FILE_JSON_MAN = Path("data/raw/api/wisatawan_mancanegara_1982_2008.json")

def inspeksi_json(path, nama):
    print("\n" + "="*60)
    print("INSPEKSI:", nama)
    print("="*60)

    if not path.exists():
        print("File tidak ditemukan:", path)
        return

    print("Ukuran file (MB):", round(path.stat().st_size / (1024**2), 3))

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("\nKEY UTAMA:")
    print(list(data.keys()))

    print("\nSTATUS:")
    print(data.get("status"))

    print("\nAVAILABILITY:")
    print(data.get("data-availability"))

    isi = data.get("data")

    print("\nTIPE DATA:")
    print(type(isi).__name__)

    if isinstance(isi, dict):
        print("\nKEY DATA:")
        print(list(isi.keys()))

        for key, value in isi.items():
            print(f"\n--- {key} ---")
            print("Tipe:", type(value).__name__)

            if isinstance(value, str):
                print("Panjang karakter:", len(value))
                print("Preview:", value[:500])

            elif isinstance(value, (list, dict)):
                print("Jumlah item:", len(value))
                print("Preview:", str(value)[:500])

            else:
                print("Nilai:", value)

    elif isinstance(isi, list):
        print("Jumlah item:", len(isi))
        print("Preview:", str(isi[:2])[:1000])

    return data


# Periksa dataset domestik
JSON_DOM = inspeksi_json(
    FILE_JSON_DOM,
    "WISATAWAN DOMESTIK"
)

# Periksa dataset mancanegara
JSON_MAN = inspeksi_json(
    FILE_JSON_MAN,
    "WISATAWAN MANCANEGARA"
)



INSPEKSI: WISATAWAN DOMESTIK
Ukuran file (MB): 0.058

KEY UTAMA:
['status', 'data-availability', 'data']

STATUS:
OK

AVAILABILITY:
available

TIPE DATA:
dict

KEY DATA:
['table_id', 'sub_id', 'subcsa_id', 'subcsa', 'title', 'table', 'cr_date', 'updt_date', 'size', 'excel', 'mapping', 'related']

--- table_id ---
Tipe: int
Nilai: 29

--- sub_id ---
Tipe: int
Nilai: 16

--- subcsa_id ---
Tipe: int
Nilai: 561

--- subcsa ---
Tipe: str
Panjang karakter: 10
Preview: Pariwisata

--- title ---
Tipe: str
Panjang karakter: 55
Preview: Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004-2025

--- table ---
Tipe: str
Panjang karakter: 57567
Preview: <html>

<head>

<style id="Wisdom_31622_Styles">
<!--table
	{mso-displayed-decimal-separator:"\,";
	mso-displayed-thousand-separator:"\.";}
.xl6631622
	{padding-top:1px;
	padding-right:1px;
	padding-left:1px;
	mso-ignore:padding;
	color:windowtext;
	font-size:10.0pt;
	font-weight:700;
	font-style:normal;
	text-decoration:none;
	font-family:Calibri, s

In [47]:

# ===== 3.4D EKSTRAKSI HTML BPS MENJADI DATAFRAME =====

import json
import pandas as pd
from pathlib import Path
from io import StringIO

# Folder sumber JSON dan folder hasil ekstraksi
RAW_API = Path("data/raw/api")
PROC_API = Path("data/processed/api")
PROC_API.mkdir(parents=True, exist_ok=True)

# File JSON hasil akuisisi API
DAFTAR_JSON = {
    "domestik": RAW_API / "wisatawan_domestik_2004_2025.json",
    "mancanegara": RAW_API / "wisatawan_mancanegara_1982_2008.json"
}

HASIL_EKSTRAKSI = {}

for nama, file_json in DAFTAR_JSON.items():

    print("\n" + "=" * 60)
    print(f"EKSTRAKSI DATA: {nama.upper()}")
    print("=" * 60)

    if not file_json.exists():
        print(f"File tidak ditemukan: {file_json}")
        continue

    # 1. Baca JSON
    with open(file_json, "r", encoding="utf-8") as f:
        json_data = json.load(f)

    # 2. Ambil bagian data
    data_detail = json_data.get("data", {})
    html_tabel = data_detail.get("table")

    if not html_tabel:
        print("HTML tabel tidak ditemukan.")
        continue

    print("Panjang HTML:", len(html_tabel), "karakter")

    # 3. Ekstrak seluruh tabel HTML
    try:
        daftar_df = pd.read_html(StringIO(html_tabel))

    except ValueError as e:
        print("HTML tidak dapat dibaca sebagai tabel:", e)
        continue

    if len(daftar_df) == 0:
        print("Tidak ditemukan tabel HTML.")
        continue

    print("Jumlah tabel HTML ditemukan:", len(daftar_df))

    # 4. Simpan semua tabel hasil ekstraksi
    HASIL_EKSTRAKSI[nama] = daftar_df

    for i, df in enumerate(daftar_df, start=1):

        nama_output = f"{nama}_tabel_{i:02d}.csv"
        path_output = PROC_API / nama_output

        df.to_csv(path_output, index=False, encoding="utf-8-sig")

        print(f"\nTabel {i}")
        print("Jumlah baris:", len(df))
        print("Jumlah kolom:", len(df.columns))
        print("File:", path_output)

        # Preview hanya 5 baris pertama
        display(df.head(5))


# 5. Ringkasan ekstraksi
print("\n" + "=" * 60)
print("RINGKASAN EKSTRAKSI")
print("=" * 60)

for nama, daftar_df in HASIL_EKSTRAKSI.items():

    print(f"\nDataset: {nama}")

    for i, df in enumerate(daftar_df, start=1):
        print(
            f"Tabel {i}: "
            f"{len(df)} baris x {len(df.columns)} kolom"
        )

print("\nFolder hasil ekstraksi:", PROC_API.resolve())



EKSTRAKSI DATA: DOMESTIK
Panjang HTML: 57567 karakter
Jumlah tabel HTML ditemukan: 1

Tabel 1
Jumlah baris: 53
Jumlah kolom: 10
File: data/processed/api/domestik_tabel_01.csv


,0,1,2,3,4,5,6,7,8,9
0,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,NaN,NaN,NaN,NaN,NaN
1,"Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...",NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bulan,2004,2005,2006,2007,2008,2009,2010,2011,2012
4,Januari,167 106,174 515,202 857,181 266,225 955,264 915,349 575,280 588,333 199



EKSTRAKSI DATA: MANCANEGARA
Panjang HTML: 51761 karakter
Jumlah tabel HTML ditemukan: 1

Tabel 1
Jumlah baris: 38
Jumlah kolom: 15
File: data/processed/api/mancanegara_tabel_01.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bulan / Month,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995
4,Januari / January,11 961,12 024,17 062,14 897,19 378,22 870,29 251,31 755,40 066,34 919,46 877,75 558,73 776,87 705



RINGKASAN EKSTRAKSI

Dataset: domestik
Tabel 1: 53 baris x 10 kolom

Dataset: mancanegara
Tabel 1: 38 baris x 15 kolom

Folder hasil ekstraksi: /content/data/processed/api


In [48]:
# ===== 3.4E VERIFIKASI HASIL EKSTRAKSI DATA BPS =====

from pathlib import Path
import pandas as pd

FOLDER_DATA = Path("data/processed/api")

daftar_csv = sorted(FOLDER_DATA.glob("*.csv"))

if not daftar_csv:
    print("BELUM ADA FILE CSV HASIL EKSTRAKSI.")
else:
    print(f"Jumlah file CSV ditemukan: {len(daftar_csv)}")

    for file in daftar_csv:
        print("\n" + "=" * 70)
        print(f"VERIFIKASI FILE: {file.name}")
        print("=" * 70)

        df = pd.read_csv(file)

        print("Jumlah baris :", df.shape[0])
        print("Jumlah kolom :", df.shape[1])
        print("\nNama kolom:")
        print(df.columns.tolist())

        print("\nJumlah missing value per kolom:")
        print(df.isna().sum())

        print("\n5 baris pertama:")
        display(df.head())

        print("\n5 baris terakhir:")
        display(df.tail())

        print("\nInformasi tipe data:")
        df.info()

Jumlah file CSV ditemukan: 2

VERIFIKASI FILE: domestik_tabel_01.csv
Jumlah baris : 53
Jumlah kolom : 10

Nama kolom:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

Jumlah missing value per kolom:
0     4
1     4
2     4
3     6
4     6
5    22
6    22
7    22
8    22
9    22
dtype: int64

5 baris pertama:


,0,1,2,3,4,5,6,7,8,9
0,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,Kunjungan Wisatawan Domestik ke Bali per Bula...,NaN,NaN,NaN,NaN,NaN
1,"Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...","Number of Domestic Visitor to Bali by Month, ...",NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bulan,2004,2005,2006,2007,2008,2009,2010,2011,2012
4,Januari,167 106,174 515,202 857,181 266,225 955,264 915,349 575,280 588,333 199



5 baris terakhir:


,0,1,2,3,4,5,6,7,8,9
48,BALI,8 052 974,9 877 911,10 120 786,9 612 511,NaN,NaN,NaN,NaN,NaN
49,Pertumbuhan,8721,2266,246,-502,NaN,NaN,NaN,NaN,NaN
50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,Source : Survey Result by Bali Government Tou...,Source : Survey Result by Bali Government Tou...,Source : Survey Result by Bali Government Tou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,Sumber : Hasil Survei Dinas Pariwisata Provin...,Sumber : Hasil Survei Dinas Pariwisata Provin...,Sumber : Hasil Survei Dinas Pariwisata Provin...,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       49 non-null     object
 1   1       49 non-null     object
 2   2       49 non-null     object
 3   3       47 non-null     object
 4   4       47 non-null     object
 5   5       31 non-null     object
 6   6       31 non-null     object
 7   7       31 non-null     object
 8   8       31 non-null     object
 9   9       31 non-null     object
dtypes: object(10)
memory usage: 4.3+ KB

VERIFIKASI FILE: mancanegara_tabel_01.csv
Jumlah baris : 38
Jumlah kolom : 15

Nama kolom:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']

Jumlah missing value per kolom:
0      4
1      4
2      5
3      6
4      7
5      8
6      8
7      8
8      8
9      8
10     8
11     8
12     8
13     8
14    23
dtype: int64

5 baris pertama:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,Banyaknya Wisatawan Mancanegara Bulanan ke Ba...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...","Number of Monthly Foreign Visitor to Bali, 19...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bulan / Month,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995
4,Januari / January,11 961,12 024,17 062,14 897,19 378,22 870,29 251,31 755,40 066,34 919,46 877,75 558,73 776,87 705



5 baris terakhir:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
33,Jumlah / Total :,1 138 895,1 230 316,1 187 153,1 355 799,1 412 839,1 356 774,1 285 842,993 185,1 472 190,1 388 984,1 262 537,1 668 531,2 085 084,NaN
34,Pertumbuhan / Growth (%),1231,743,-910,3216,421,-397,-523,-2276,4823,-565,-910,3216,2497,NaN
35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,Sumber : Badan Pusat Statistik Provinsi Bali,Sumber : Badan Pusat Statistik Provinsi Bali,Sumber : Badan Pusat Statistik Provinsi Bali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,Source : BPS - Statistics of Bali Province,Source : BPS - Statistics of Bali Province,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       34 non-null     object
 1   1       34 non-null     object
 2   2       33 non-null     object
 3   3       32 non-null     object
 4   4       31 non-null     object
 5   5       30 non-null     object
 6   6       30 non-null     object
 7   7       30 non-null     object
 8   8       30 non-null     object
 9   9       30 non-null     object
 10  10      30 non-null     object
 11  11      30 non-null     object
 12  12      30 non-null     object
 13  13      30 non-null     object
 14  14      15 non-null     object
dtypes: object(15)
memory usage: 4.6+ KB


In [49]:

# ==========================================
# MEMBUAT ZIP FILE PROYEK UTS DAM
# ==========================================

import os
import zipfile

# Lokasi folder proyek
folder_proyek = "/content"

# Nama file ZIP yang akan dibuat
nama_zip = "/content/UTS-DAM-Pariwisata-Bali.zip"

# Folder dan file yang ingin dimasukkan
folder_target = [
    "data",
    "docs"
]

file_target = [
    "README.md"
]

# Membuat ZIP
with zipfile.ZipFile(nama_zip, "w", zipfile.ZIP_DEFLATED) as zipf:

    # Memasukkan folder
    for folder in folder_target:
        path_folder = os.path.join(folder_proyek, folder)

        if os.path.exists(path_folder):
            for root, dirs, files in os.walk(path_folder):
                for file in files:
                    path_file = os.path.join(root, file)
                    arcname = os.path.relpath(path_file, folder_proyek)
                    zipf.write(path_file, arcname)

    # Memasukkan file README
    for file in file_target:
        path_file = os.path.join(folder_proyek, file)

        if os.path.exists(path_file):
            zipf.write(path_file, file)

print("ZIP berhasil dibuat:", nama_zip)
print("Ukuran ZIP:", round(os.path.getsize(nama_zip) / (1024**2), 2), "MB")


ZIP berhasil dibuat: /content/UTS-DAM-Pariwisata-Bali.zip
Ukuran ZIP: 0.81 MB


In [50]:
import zipfile

file_zip = "/content/UTS-DAM-Pariwisata-Bali.zip"

with zipfile.ZipFile(file_zip, "r") as zip_ref:
    print("ISI FILE ZIP:")
    for nama_file in zip_ref.namelist():
        print(nama_file)

ISI FILE ZIP:
data/processed/panel_wisatawan_bali_cuaca_1982_2025.metadata.json
data/processed/panel_wisatawan_bali_cuaca_1982_2025.csv
data/processed/wisatawan_bali_long_gabungan.csv
data/processed/wilayah_bali_wikipedia.csv
data/processed/cuaca_denpasar_bulanan_1982_2025.csv
data/processed/api/mancanegara_tabel_01.csv
data/processed/api/domestik_tabel_01.csv
data/metadata/provenance.csv
data/metadata/data_ethics_checklist_draf.csv
data/metadata/Data_Ethics_Checklist_BPS_Wisatawan_Bali.csv
data/metadata/data_ethics_checklist_lengkap.csv
data/raw/api/openmeteo_denpasar_2018_2021.json
data/raw/api/wisatawan_domestik_2004_2025.json
data/raw/api/openmeteo_denpasar_1990_1993.json
data/raw/api/openmeteo_denpasar_2014_2017.json
data/raw/api/openmeteo_denpasar_2006_2009.json
data/raw/api/openmeteo_denpasar_1998_2001.json
data/raw/api/openmeteo_denpasar_1986_1989.json
data/raw/api/openmeteo_denpasar_1982_1985.json
data/raw/api/wisatawan_mancanegara_1982_2008.json
data/raw/api/bps_webapi_var_wi

In [51]:
from google.colab import files

files.download("/content/UTS-DAM-Pariwisata-Bali.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>